# 11 — Attack/Defense Elo backtest

Two ratings per team (attack + defense) vs single Elo. Tests:
1. **Asymmetric features in logit/MLP/XGB**: do `attack_diff` and `defense_diff` beat single `elo_diff`?
2. **Standalone A/D Elo as goal model**: compute Poisson probs from A/D-implied goal rates and score directly.

In [1]:
import sys
sys.path.append('..')
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import log_loss
import xgboost as xgb
from scipy.stats import poisson

from src.elo import EloModel
from src.attack_defense_elo import AttackDefenseEloModel
from src.draw_model import match_outcome

## Compute A/D Elo pre-match ratings on full history

In [2]:
df_all = pd.read_csv('../data/processed/matches_competitive.csv', parse_dates=['date']).dropna(subset=['home_score','away_score'])
df_sv  = pd.read_csv('../data/processed/matches_with_squad_value.csv', parse_dates=['date'])

elo = EloModel()
elo_enriched = elo.fit(df_all)
ad = AttackDefenseEloModel()
ad_enriched = ad.fit(df_all)

# Merge both into the squad-value frame
df = df_sv.merge(
    elo_enriched[['date','home_team','away_team','home_elo_pre','away_elo_pre']],
    on=['date','home_team','away_team'], how='left')
df = df.merge(
    ad_enriched[['date','home_team','away_team',
                  'home_attack_pre','home_defense_pre','away_attack_pre','away_defense_pre',
                  'exp_home_goals','exp_away_goals']],
    on=['date','home_team','away_team'], how='left')
df = df.dropna(subset=['home_elo_pre','away_elo_pre','home_attack_pre'])

HOME_ADVANTAGE = 100; FLOOR = 1e5
df['neutral'] = df['neutral'].fillna(False)
df['eff_elo_diff']  = (df['home_elo_pre'] + (~df['neutral']).astype(int)*HOME_ADVANTAGE) - df['away_elo_pre']
df['elo_diff_norm'] = df['eff_elo_diff'] / 400.0
df['log_value_diff'] = np.log10(df['home_top_n_value_eur'].clip(lower=FLOOR) / df['away_top_n_value_eur'].clip(lower=FLOOR))
df['outfield_age_diff'] = df['home_outfield_mean_age'] - df['away_outfield_mean_age']
df['top1_share_diff']   = df['home_top1_share'] - df['away_top1_share']
df['fifa_points_diff']  = (df['home_fifa_points'].fillna(0) - df['away_fifa_points'].fillna(0)) / 1000.0

# A/D Elo derived features. Scale to similar order as elo_diff_norm.
df['attack_vs_def_h']  = (df['home_attack_pre']  - df['away_defense_pre']) / 200.0   # home goal-rate advantage
df['attack_vs_def_a']  = (df['away_attack_pre']  - df['home_defense_pre']) / 200.0   # away goal-rate advantage
df['expected_goal_diff'] = (df['exp_home_goals'] - df['exp_away_goals'])

df['outcome'] = df.apply(lambda r: match_outcome(r['home_score'], r['away_score']), axis=1)

FEAT_BASE = ['elo_diff_norm','log_value_diff','outfield_age_diff','top1_share_diff','fifa_points_diff']
FEAT_AD   = FEAT_BASE + ['attack_vs_def_h','attack_vs_def_a']
FEAT_AD_ALT = FEAT_BASE + ['expected_goal_diff']
needed = FEAT_AD_ALT + ['outcome']
valid = df.dropna(subset=needed)
valid = valid[(valid['home_top_n_value_eur']>FLOOR)&(valid['away_top_n_value_eur']>FLOOR)].copy()
print(f'Training-eligible matches: {len(valid):,}')
print(valid[['elo_diff_norm','attack_vs_def_h','attack_vs_def_a','expected_goal_diff']].describe().round(3))

Training-eligible matches: 7,641
       elo_diff_norm  attack_vs_def_h  attack_vs_def_a  expected_goal_diff
count       7641.000         7641.000         7641.000            7641.000
mean           0.207           -0.235           -0.258               0.280
std            0.434            0.345            0.346               0.898
min           -1.601           -1.990           -1.792              -5.904
25%           -0.075           -0.443           -0.473              -0.213
50%            0.195           -0.236           -0.263               0.240
75%            0.479           -0.025           -0.048               0.732
max            2.077            1.478            1.496               7.643


## Walk-forward backtest — A/D features added to the FEAT_WITH_FIFA baseline

In [3]:
def score(y_true, proba):
    p = np.clip(proba, 1e-6, 1-1e-6)
    oh = np.zeros_like(p); oh[np.arange(len(y_true)), y_true] = 1
    return {
        'log_loss': log_loss(y_true, p, labels=[0,1,2]),
        'accuracy': float((p.argmax(axis=1) == y_true).mean()),
        'brier':    float(np.mean(np.sum((p-oh)**2, axis=1))),
    }

def fit_logit(X, y):
    return LogisticRegression(max_iter=2000).fit(X, y)
def fit_xgb(X, y):
    return xgb.XGBClassifier(objective='multi:softprob', num_class=3, n_estimators=300,
                             max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                             reg_lambda=1.0, eval_metric='mlogloss', n_jobs=4, verbosity=0).fit(X, y)
def fit_mlp(X_scaled, y):
    return MLPClassifier(hidden_layer_sizes=(32,16), activation='relu', max_iter=500,
                          learning_rate_init=0.005, alpha=1e-3, early_stopping=True, random_state=0).fit(X_scaled, y)
def proba_ord(clf, X):
    p = clf.predict_proba(X); order = [list(clf.classes_).index(c) for c in (0,1,2)]
    return p[:, order]

rows = []
for year in [2010, 2014, 2018, 2022]:
    train = valid[valid['date'].dt.year < year]
    test  = valid[(valid['date'].dt.year == year) & (valid['tournament']=='FIFA World Cup')]
    if test.empty: continue
    y_train = train['outcome'].to_numpy(); y_test = test['outcome'].to_numpy()
    row = {'year': year, 'n_test': len(test)}
    for feat_name, cols in [('base', FEAT_BASE), ('+ad', FEAT_AD), ('+eg_diff', FEAT_AD_ALT)]:
        sc = StandardScaler().fit(train[cols].to_numpy())
        Xtr = train[cols].to_numpy(); Xte = test[cols].to_numpy()
        for model_name, clf in [
            ('logit', fit_logit(Xtr, y_train)),
            ('xgb',   fit_xgb(Xtr, y_train)),
            ('mlp',   fit_mlp(sc.transform(Xtr), y_train)),
        ]:
            X_pred = sc.transform(Xte) if model_name == 'mlp' else Xte
            p = proba_ord(clf, X_pred)
            s = score(y_test, p)
            k = f'{model_name}_{feat_name}'
            row[f'{k}_ll'] = s['log_loss']
            row[f'{k}_acc'] = s['accuracy']
    rows.append(row)
res = pd.DataFrame(rows)
print('Averages:')
for name in ['logit_base','logit_+ad','logit_+eg_diff','xgb_base','xgb_+ad','xgb_+eg_diff','mlp_base','mlp_+ad','mlp_+eg_diff']:
    print(f"  {name:18s}  log_loss: {res[f'{name}_ll'].mean():.4f}   accuracy: {res[f'{name}_acc'].mean():.4f}")

Averages:
  logit_base          log_loss: 1.0074   accuracy: 0.5273
  logit_+ad           log_loss: 1.0164   accuracy: 0.5117
  logit_+eg_diff      log_loss: 1.0160   accuracy: 0.5078
  xgb_base            log_loss: 1.0025   accuracy: 0.5273
  xgb_+ad             log_loss: 0.9982   accuracy: 0.5156
  xgb_+eg_diff        log_loss: 1.0162   accuracy: 0.5156
  mlp_base            log_loss: 1.0111   accuracy: 0.5391
  mlp_+ad             log_loss: 1.0162   accuracy: 0.5156
  mlp_+eg_diff        log_loss: 1.0034   accuracy: 0.5195


## Standalone A/D Elo as a goal model

Use the implied expected goals directly as Poisson rates → outcome probs. No classifier.

In [4]:
def adelo_proba(lam_h, lam_a, max_goals=10):
    x = np.arange(max_goals+1)
    px = poisson.pmf(x, lam_h); py = poisson.pmf(x, lam_a)
    m = np.outer(px, py); tot = m.sum()
    return np.array([np.triu(m,1).sum(), np.diag(m).sum(), np.tril(m,-1).sum()]) / tot

rows2 = []
for year in [2010, 2014, 2018, 2022]:
    test = valid[(valid['date'].dt.year==year) & (valid['tournament']=='FIFA World Cup')]
    if test.empty: continue
    probs = np.array([adelo_proba(r['exp_home_goals'], r['exp_away_goals']) for _, r in test.iterrows()])
    s = score(test['outcome'].to_numpy(), probs)
    rows2.append({'year': year, 'n': len(test), 'log_loss': s['log_loss'], 'accuracy': s['accuracy'], 'brier': s['brier']})
res2 = pd.DataFrame(rows2)
print(res2.round(4).to_string(index=False))
print(f"\nA/D Elo standalone   log_loss: {res2['log_loss'].mean():.4f}   accuracy: {res2['accuracy'].mean():.4f}   brier: {res2['brier'].mean():.4f}")

 year  n  log_loss  accuracy  brier
 2010 64    1.0505    0.4688 0.6328
 2014 64    1.0508    0.5000 0.6354
 2018 64    1.0330    0.5156 0.6219
 2022 64    1.0907    0.4688 0.6578

A/D Elo standalone   log_loss: 1.0562   accuracy: 0.4883   brier: 0.6370
